To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your own computer, follow the installation instructions on our Github page [here](https://docs.unsloth.ai/get-started/installing-+-updating).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)


### News

**Read our [Gemma 3 blog](https://unsloth.ai/blog/gemma3) for what's new in Unsloth and our [Reasoning blog](https://unsloth.ai/blog/r1-reasoning) on how to train reasoning models.**

Visit our docs for all our [model uploads](https://docs.unsloth.ai/get-started/all-our-models) and [notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).


### Installation

In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    !pip install --no-deps unsloth vllm

In [ ]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    !pip install --no-deps unsloth vllm
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    # Skip restarting message in Colab
    import sys, re, requests; modules = list(sys.modules.keys())
    for x in modules: sys.modules.pop(x) if "PIL" in x or "google" in x else None
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft "trl==0.15.2" triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer

    # vLLM requirements - vLLM breaks Colab due to reinstalling numpy
    f = requests.get("https://raw.githubusercontent.com/vllm-project/vllm/refs/heads/main/requirements/common.txt").content
    with open("vllm_requirements.txt", "wb") as file:
        file.write(re.sub(rb"(transformers|numpy|xformers)[^\n]{1,}\n", b"", f))
    !pip install -r vllm_requirements.txt

### Unsloth

Load up `Qwen 2.5 3B Instruct`, and set parameters

In [ ]:
!pip install vllm==0.8.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.6/293.6 MB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.9/97.9 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 115.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 105.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 MB 34.6 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: gguf
    Found existing installation: gguf 0.14.0
    Uninstalling gguf-0.14.0:
      Successfully uninstalled gguf-0.14.0
  Attempting uninstall: xformers
    Found existing installation: xforme

In [ ]:
pip list

Package                            Version
---------------------------------- -------------------
absl-py                            1.4.0
accelerate                         1.5.2
aiohappyeyeballs                   2.6.1
aiohttp                            3.11.15
aiosignal                          1.3.2
airportsdata                       20250224
alabaster                          1.0.0
albucore                           0.0.23
albumentations                     2.0.5
ale-py                             0.10.2
altair                             5.5.0
annotated-types                    0.7.0
anyio                              4.9.0
argon2-cffi                        23.1.0
argon2-cffi-bindings               21.2.0
array_record                       0.7.1
arviz                              0.21.0
astor                              0.8.1
astropy                            7.0.1
astropy-iers-data                  0.2025.3.31.0.36.18
astunparse                         1.6.3
atpublic         

In [ ]:
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HUGGINGFACE_TOKEN')
login(hf_token)

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch
max_seq_length = 5120 # Can increase for longer reasoning traces
lora_rank = 8 # Larger rank = smarter, but slower

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/DeepSeek-R1-Distill-Qwen-7B",
    max_seq_length = max_seq_length,
    load_in_4bit = True, # False for LoRA 16bit
    fast_inference = True, # Enable vLLM fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.7, # Reduce if out of memory
)


model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj"
        ], # Remove QKVO if out of memory
    lora_alpha = lora_rank,
    use_gradient_checkpointing = "unsloth", # Enable long context finetuning
    random_state = 3407,
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 04-10 10:24:50 [__init__.py:239] Automatically detected platform cuda.
==((====))==  Unsloth 2025.3.19: Fast Qwen2 patching. Transformers: 4.50.3. vLLM: 0.8.2.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/deepseek-r1-distill-qwen-7b-unsloth-bnb-4bit with actual GPU utilization = 69.2%
Unsloth: Your GPU has CUDA compute capability 8.0 with VRAM = 39.56 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 5120. Num Sequences = 288.
Unsloth: vLLM's KV Cache can u

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.52G [00:00<?, ?B/s]

INFO 04-10 10:25:31 [weight_utils.py:281] Time spent downloading weights for unsloth/deepseek-r1-distill-qwen-7b-unsloth-bnb-4bit: 21.200132 seconds


model.safetensors.index.json:   0%|          | 0.00/100k [00:00<?, ?B/s]

...

INFO 04-10 10:26:49 [model_runner.py:1570] Graph capturing finished in 58 secs, took 0.69 GiB
INFO 04-10 10:26:49 [llm_engine.py:447] init engine (profile, create kv cache, warmup model) took 70.35 seconds


tokenizer_config.json:   0%|          | 0.00/6.78k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2025.3.19 patched 28 layers with 28 QKV layers, 28 O layers and 0 MLP layers.


### Data Prep
<a name="Data"></a>

We directly leverage [@willccbb](https://gist.github.com/willccbb/4676755236bb08cab5f4e54a0475d6fb) for data prep and all reward functions. You are free to create your own!

In [ ]:
import re
from datasets import load_dataset, Dataset

# Load and prep dataset
SYSTEM_PROMPT = """
Respond in the following format:
<reasoning>
...
</reasoning>
<answer>
...
</answer>
"""

XML_COT_FORMAT = """\
<reasoning>
{reasoning}
</reasoning>
<answer>
{answer}
</answer>
"""

def extract_xml_answer(text: str) -> str:
    answer = text.split("<answer>")[-1]
    answer = answer.split("</answer>")[0]
    return answer.strip()

def extract_hash_answer(text: str) -> str | None:
    if "####" not in text:
        return None
    return text.split("####")[1].strip()

# uncomment middle messages for 1-shot prompting
def get_gsm8k_questions(split = "train") -> Dataset:
    data = load_dataset('openai/gsm8k', 'main')[split] # type: ignore
    data = data.map(lambda x: { # type: ignore
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': x['question']}
        ],
        'answer': extract_hash_answer(x['answer'])
    }) # type: ignore
    return data # type: ignore

dataset = get_gsm8k_questions()

# Reward functions
def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    responses = [completion[0]['content'] for completion in completions]
    q = prompts[0][-1]['content']
    extracted_responses = [extract_xml_answer(r) for r in responses]
    print('-'*20, f"Question:\n{q}", f"\nAnswer:\n{answer[0]}", f"\nResponse:\n{responses[0]}", f"\nExtracted:\n{extracted_responses[0]}")
    return [2.0 if r == a else 0.0 for r, a in zip(extracted_responses, answer)]

def int_reward_func(completions, **kwargs) -> list[float]:
    responses = [completion[0]['content'] for completion in completions]
    extracted_responses = [extract_xml_answer(r) for r in responses]
    return [0.5 if r.isdigit() else 0.0 for r in extracted_responses]

def strict_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has a specific format."""
    pattern = r"^<reasoning>\n.*?\n</reasoning>\n<answer>\n.*?\n</answer>\n$"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r) for r in responses]
    return [0.5 if match else 0.0 for match in matches]

def soft_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has a specific format."""
    pattern = r"<reasoning>.*?</reasoning>\s*<answer>.*?</answer>"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r) for r in responses]
    return [0.5 if match else 0.0 for match in matches]

def count_xml(text) -> float:
    count = 0.0
    if text.count("<reasoning>\n") == 1:
        count += 0.125
    if text.count("\n</reasoning>\n") == 1:
        count += 0.125
    if text.count("\n<answer>\n") == 1:
        count += 0.125
        count -= len(text.split("\n</answer>\n")[-1])*0.001
    if text.count("\n</answer>") == 1:
        count += 0.125
        count -= (len(text.split("\n</answer>")[-1]) - 1)*0.001
    return count

def xmlcount_reward_func(completions, **kwargs) -> list[float]:
    contents = [completion[0]["content"] for completion in completions]
    return [count_xml(c) for c in contents]

README.md:   0%|          | 0.00/7.94k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

# **自定义数据集 prompt设计+reward设计**

In [ ]:
from datasets import load_dataset,Dataset
import json
import re

In [ ]:
SYSTEM_PROMPT = """
你是一个儿童故事分析专家，请从文本中精准提取所有事件及其关系。文本中`<sen>`表示句子分隔符，需分别分析每个句子。

**事件定义**
每行一个事件，格式为：(触发词；主语；宾语；时间状语；地点状语)
注意：主语/宾语多个时用逗号分隔，如：小狗,小男孩；缺失字段填"无"

**关系定义**
每行一个关系，格式为：[源事件五元组] [目标事件五元组] [关系类型]
支持的关系类型：并列、动机-因果、心理-因果、物理-因果、使能-因果
注意：事件必须来源上一步抽取出的事件

**输出格式**
严格按以下结构输出：
<事件列表>
(触发词；主语；宾语；时间；地点)
...
</事件列表>
<关系列表>
[源事件五元组] [目标事件五元组] [关系类型]
...
</关系列表>

**示例**
输入：夜晚小孩子和狗全睡着了。<sen>那个青蛙趁夜色逃跑了。
输出：
<事件列表>
(睡着；小孩子，狗；无；夜晚；无)
(逃跑；青蛙；无；无；无)
</事件列表>
<关系列表>
(睡着；小孩子，狗；无；夜晚；无) (逃跑；青蛙；无；无；无) 使能-因果
</关系列表>
"""


In [ ]:
def load_dataset(jsonl_path):
    data = []
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            events = item["event"]
            relations = item["relation"]
            # 构建事件列表
            event_list = "<事件列表>\n"
            for event in events:
                event_list += event + "\n"
            event_list += "</事件列表>"

            # 构建关系列表
            relation_list = "<关系列表>\n"
            for relation in relations:
                relation_list += relation + "\n"
            relation_list += "</关系列表>"

            answer = event_list+relation_list
            data.append({
                "text": item["text"],
                "answer": answer
            })
    return data

In [ ]:
dataset = load_dataset("/content/train_data.jsonl")
train_data = [{
    "prompt": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": item["text"]}
    ],
    "answer": item["answer"]
} for item in dataset]

In [ ]:
print(train_data[0])

{'prompt': [{'role': 'system', 'content': '\n你是一个儿童故事分析专家，请从文本中精准提取所有事件及其关系。文本中`<sen>`表示句子分隔符，需分别分析每个句子。\n\n**事件定义**\n每行一个事件，格式为：(触发词；主语；宾语；时间状语；地点状语)\n注意：主语/宾语多个时用逗号分隔，如：小狗,小男孩；缺失字段填"无"\n\n**关系定义**\n每行一个关系，格式为：[源事件五元组] [目标事件五元组] [关系类型]\n支持的关系类型：并列、动机-因果、心理-因果、物理-因果、使能-因果\n注意：事件必须来源上一步抽取出的事件\n\n**输出格式**\n严格按以下结构输出：\n<事件列表>\n(触发词；主语；宾语；时间；地点)\n...\n</事件列表>\n<关系列表>\n[源事件五元组] [目标事件五元组] [关系类型]\n...\n</关系列表>\n\n**示例**\n输入：夜晚小孩子和狗全睡着了。<sen>那个青蛙趁夜色逃跑了。\n输出：\n<事件列表>\n(睡着；小孩子，狗；无；夜晚；无)\n(逃跑；青蛙；无；无；无)\n</事件列表>\n<关系列表>\n(睡着；小孩子，狗；无；夜晚；无) (逃跑；青蛙；无；无；无) 使能-因果\n</关系列表>\n'}, {'role': 'user', 'content': '青蛙在瓶子里的时候.<sen>他就睡了会儿觉.<sen>青蛙跑出来以后.<sen>他就睡醒以后.<sen>我的青蛙呢.<sen>他就生气了.<sen>他出门找找青蛙在哪里.<sen>他叫,青蛙.<sen>他就想了一下青蛙在哪里.<sen>狗就打碎了瓶子.<sen>他就生气了.<sen>他叫,青蛙.<sen>这个树林里它就没青蛙了.<sen>这里他就找到青蛙了.<sen>青蛙呢?<sen>他就到上面.<sen>也没有青蛙.<sen>咕咚.<sen>全摔下来了.<sen>小狗也要跑啦.<sen>他就说,我的小狗呢.<sen>我的小狗也跑了.<sen>他就当成梅花鹿了.<sen>他就把梅花鹿.<sen>梅花鹿把他给摔下来了.<sen>摔到了水里了.<sen>他就坐在泥坑上.<sen>小狗在哪里?<sen>他就找到青蛙了.<sen>青蛙在这里.<sen>快给我跳过来.'}], 'answer': '<事件

In [ ]:
dataset_train_mini=train_data[:100]

In [ ]:
dataset_mini=Dataset.from_list(dataset_train_mini)

In [ ]:
print(len(dataset_mini))

378


In [ ]:
max_token_length = 0

for example in dataset_mini:
    # 使用 tokenizer 将文本转化为 token 列表
    # prompt = example['prompt']
    # content = prompt[0]['content']
    tokens = tokenizer(example['answer'], truncation=True, padding=False)  # 只进行 token 化，不进行填充
    token_length = len(tokens['input_ids'])  # 获取 token 长度
    max_token_length = max(max_token_length, token_length)  # 更新最大 token 长度

print(f"Max token sequence length: {max_token_length}")

Max token sequence length: 3687


**设计提取范式，从模型回答中提取XML标签下的两部分内容**

In [ ]:
import re
from typing import Dict, List

In [ ]:
def parse_model_output(output: str) -> Dict:
    """解析模型生成的事件和关系"""
    result = {"events": {}, "relations": []}

    # 解析事件
    event_block = re.search(r"<事件列表>(.*?)</事件列表>", output, re.DOTALL)
    if event_block:
        for line in event_block.group(1).split('\n'):
            line = line.strip()
            if match := re.match(r"EV\d{3}\s+(\(.*?\))", line):
                event_id = line.split()[0]
                result["events"][event_id] = line

    # 解析关系
    rel_block = re.search(r"<关系列表>(.*?)</关系列表>", output, re.DOTALL)
    if rel_block:
        for line in rel_block.group(1).split('\n'):
            parts = line.strip().split()
            if len(parts) ==3:
                result["relations"].append({
                    "source": parts[0],
                    "target": parts[1],
                    "type": parts[2]
                })
    return result

**奖励函数**

In [ ]:
!pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 29.7 MB/s eta 0:00:00


In [ ]:
from rapidfuzz import fuzz
import re
import numpy as np
from collections import defaultdict
from scipy.optimize import linear_sum_assignment

In [ ]:
def format_reward_func(completions, **kwargs) -> list[float]:
    """纯格式验证奖励函数（事件+关系）"""
    # 预编译正则表达式
    EVENT_PATTERN = re.compile(r"\([^；]+；[^；]+；[^；]+；[^；]+；[^；]+\)")
    REL_PATTERN = re.compile(
        rf"^\s*{EVENT_PATTERN.pattern}\s+"
        rf"{EVENT_PATTERN.pattern}\s+"
        r"(并列|动机-因果|心理-因果|物理-因果|使能-因果)\s*$"
    )

    rewards = []
    for completion in completions:
        content = completion[0]["content"]
        total_score = 0.0

        # --------------------------
        # 1. XML标签检查（权重30%）
        # --------------------------
        has_valid_event_tag = bool(re.search(r"<事件列表>\s*?.+?</事件列表>", content, re.DOTALL))
        has_valid_rel_tag = bool(re.search(r"<关系列表>\s*?.+?</关系列表>", content, re.DOTALL))
        tag_score = 0.3 * (0.5*has_valid_event_tag + 0.5*has_valid_rel_tag)
        total_score += tag_score

        # --------------------------
        # 2. 事件列表格式（权重50%）
        # --------------------------
        event_blocks = re.findall(r"<事件列表>(.*?)</事件列表>", content, re.DOTALL)
        event_lines = [line.strip() for block in event_blocks for line in block.split("\n") if line.strip()]

        valid_events = sum(1 for line in event_lines if EVENT_PATTERN.match(line))
        event_score = 0.5 * (valid_events / max(len(event_lines), 1))
        total_score += event_score

        # --------------------------
        # 3. 关系列表格式（权重20%）
        # --------------------------
        rel_blocks = re.findall(r"<关系列表>(.*?)</关系列表>", content, re.DOTALL)
        rel_lines = [line.strip() for block in rel_blocks for line in block.split("\n") if line.strip()]

        valid_rels = sum(1 for line in rel_lines if REL_PATTERN.match(line))
        rel_score = 0.2 * (valid_rels / max(len(rel_lines), 1))
        total_score += rel_score

        rewards.append(max(total_score, 0))

    return rewards

In [ ]:
def field_accuracy_reward_func_optimized(prompts, completions, answer, **kwargs) -> list[float]:
    """优化后的事件抽取正确性奖励函数"""
    rewards = []

    # 预编译正则提高解析速度
    event_pattern = re.compile(r"\(((?:[^()]|\(.*?\))*?)\)")

    for comp, gold_answer in zip(completions, answer):
        pred_content = comp[0]["content"]
        gold_content = gold_answer

        # --------------------------
        # 1. 解析并标准化事件
        # --------------------------
        def parse_and_normalize(content):
            events = []
            for line in re.findall(r"\(([^)]+)\)", content):
                line = line.replace(";", "；")
                parts = line.strip().split("；")
                if len(parts) !=5:
                    continue
                # 标准化处理（关键步骤）
                norm_event = {
                    "trigger": parts[0].strip(),
                    "subject": sort_comma_values(parts[1]),  # 主语排序
                    "object": sort_comma_values(parts[2]),   # 宾语排序
                    "time": parts[3].strip(),
                    "location": parts[4].strip()
                }
                events.append(norm_event)
            return events

        pred_events = parse_and_normalize(pred_content)
        gold_events = parse_and_normalize(gold_content)

        # --------------------------
        # 2. 构建相似度矩阵（核心优化）
        # --------------------------
        n_pred = len(pred_events)
        n_gold = len(gold_events)
        sim_matrix = np.zeros((n_pred, n_gold))

        # 批量计算相似度（避免重复计算）
        for i, p_event in enumerate(pred_events):
            for j, g_event in enumerate(gold_events):
                sim_matrix[i,j] = event_similarity_optimized(p_event, g_event)

        # --------------------------
        # 3. 使用匈牙利算法寻找最优匹配
        # --------------------------
        row_ind, col_ind = linear_sum_assignment(-sim_matrix)  # 最大化相似度
        matched_pairs = []
        # 计算动态阈值（基于矩阵中位数）
        valid_sims = sim_matrix[sim_matrix > 0]
        if len(valid_sims) > 0:
            dynamic_threshold = max(0.6, np.median(valid_sims)*0.9)
        else:
            dynamic_threshold = 0.6

        matched_pairs = [(r,c) for r,c in zip(row_ind, col_ind) if sim_matrix[r,c] >= dynamic_threshold]

        # --------------------------
        # 4. 计算指标
        # --------------------------
        tp = len(matched_pairs)
        fp = n_pred - tp
        fn = n_gold - tp

        # F1计算（带数量差异惩罚）
        precision = tp / (tp + fp + 1e-9)
        recall = tp / (tp + fn + 1e-9)
        count_diff_penalty = min(1.0, abs(n_pred - n_gold) / max(n_pred, n_gold, 1))  # 差异比例
        f1 = 2 * (precision * recall) / (precision + recall + 1e-9)
        f1 *= (1 - 0.3 * count_diff_penalty)  # 最大惩罚30%
        rewards.append(max(f1, 0))  # 确保非负

    return rewards

def sort_comma_values(s: str) -> str:
    """对逗号分隔的值进行排序（如'小狗，小男孩'→'小男孩，小狗'）"""
    if "，" in s:
        return "，".join(sorted(s.split("，")))
    return s

def event_similarity_optimized(e1: dict, e2: dict) -> float:
    """优化后的事件相似度计算（向量化友好）"""
    weights = np.array([0.4, 0.2, 0.2, 0.1, 0.1])
    scores = np.zeros(5)

    trigger_sim = fuzz.ratio(e1["trigger"], e2["trigger"]) / 100
    scores[0] = trigger_sim if trigger_sim >= 0.7 else 0  # 相似度≥70%才计分
    scores[1] = fuzz.ratio(e1["subject"], e2["subject"]) / 100
    scores[2] = fuzz.ratio(e1["object"], e2["object"]) / 100
    scores[3] = fuzz.ratio(e1["time"], e2["time"]) / 100
    scores[4] = fuzz.ratio(e1["location"], e2["location"]) / 100

    return np.dot(weights, scores)

In [ ]:
'''
def field_accuracy_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    rewards = []

    for completion, gold_answer in zip(completions, answer):
        pred_lines = extract_events(completion[0]["content"])
        gold_lines = extract_events(gold_answer.strip())

        # 解析事件字段（容错处理）
        pred_events = [parse_event(line) for line in pred_lines]
        gold_events = [parse_event(line) for line in gold_lines]

        # 计算事件级F1（考虑顺序）
        tp, fp, fn = 0, 0, 0
        matched_gold = set()

        # 为每个预测事件找最佳匹配
        for p_idx, p_event in enumerate(pred_events):
            if not p_event:  # 解析失败事件
                fp += 1
                continue

            best_sim, best_g_idx = -1, -1
            for g_idx, g_event in enumerate(gold_events):
                if g_idx in matched_gold or not g_event:
                    continue
                # 计算事件相似度（加权字段）
                sim = event_similarity(p_event, g_event)
                if sim > best_sim:
                    best_sim = sim
                    best_g_idx = g_idx

            if best_sim >= 0.8:  # 相似度阈值
                tp += 1
                matched_gold.add(best_g_idx)
            else:
                fp += 1

        # 计算未匹配的真实事件
        fn = len(gold_events) - len(matched_gold)

        # F1计算
        precision = tp / (tp + fp + 1e-9)
        recall = tp / (tp + fn + 1e-9)
        f1 = 2 * (precision * recall) / (precision + recall + 1e-9)

        event_diff = abs(len(pred_events) - len(gold_events))
        f1 = f1 * 0.9 - 0.1 * event_diff  # 每多/少一个事件扣0.1分

        rewards.append(f1 * 1.0)

    return rewards

def extract_events(text: str) -> list[str]:
    """提取事件列表，从<事件列表>...<\事件列表>中提取每一行事件"""
    # 使用正则去除<事件列表>和<\事件列表>标签
    text = re.sub(r'<事件列表>.*?<\事件列表>', '', text, flags=re.DOTALL)

    # 按行分割文本，提取每行的事件
    event_lines = [line.strip() for line in text.strip().split('\n') if line.strip()]
    return event_lines

def parse_event(line: str) -> dict:
    """解析单行事件，返回字段字典（解析失败返回None）"""
    try:
        # 匹配事件编号和事件内容
        event_pattern = r'\(([^)]+)\)'
        match = re.match(event_pattern, line.strip())
        if match:
            parts = match.group(1).split("；")  # 解析事件字段
            if len(parts) == 5:
                return {
                    "trigger": parts[0], "subject": parts[1],
                    "object": parts[2], "time": parts[3], "location": parts[4]
                }
        return None
    except:
        return None

def event_similarity(event1: dict, event2: dict) -> float:
    """加权字段相似度计算"""
    weights = {"trigger": 0.4, "subject": 0.2, "object": 0.2, "time": 0.1, "location": 0.1}
    total = 0
    for field, w in weights.items():
        # 处理"无"的特殊情况
        if event1[field] == "无" and event2[field] == "无":
            total += w * 0.5
        else:
            sim = fuzz.ratio(event1[field], event2[field]) / 100
            total += w * sim
    return total
'''

In [ ]:
def relation_accuracy_reward_optimized(prompts, completions, answer, **kwargs) -> List[float]:
    rewards = []
    rel_pattern = re.compile(
        r"\(((?:[^()]|\([^)]*\))+)\)\s*"  # 事件1
        r"\(((?:[^()]|\([^)]*\))+)\)\s*"  # 事件2
        r"(并列|动机-因果|心理-因果|物理-因果|使能-因果)"  # 关系类型
    )

    for comp, gold in zip(completions, answer):
        pred_content = comp[0]["content"]
        gold_content = gold

        # --------------------------
        # 1. 解析并标准化事件关系
        # --------------------------
        def parse_relations(content):
            relations = []
            for match in rel_pattern.finditer(content):
                e1 = normalize_event_str(match.group(1))
                e2 = normalize_event_str(match.group(2))
                rel_type = match.group(3).strip()
                relations.append((e1, e2, rel_type))
            return relations

        pred_rels = parse_relations(pred_content)
        gold_rels = parse_relations(gold_content)

        # --------------------------
        # 2. 构建事件索引与相似度矩阵（关键优化）
        # --------------------------
        # 收集所有唯一事件
        all_events = set()
        for e1, e2, _ in pred_rels + gold_rels:
            all_events.update([e1, e2])
        event_list = list(all_events)
        event_to_idx = {e:i for i,e in enumerate(event_list)}

        # 预计算事件相似度矩阵 (N x N)
        n = len(event_list)
        sim_matrix = np.zeros((n, n))
        for i in range(n):
            for j in range(n):
                sim_matrix[i,j] = fuzz.token_sort_ratio(event_list[i], event_list[j])/100

        # --------------------------
        # 3. 按关系类型分组加速匹配
        # --------------------------
        # 分组结构：{rel_type: [(pred_e1_idx, pred_e2_idx), ...]}
        pred_groups = defaultdict(list)
        for p_e1, p_e2, p_type in pred_rels:
            pred_groups[p_type].append((
                event_to_idx.get(p_e1, -1),
                event_to_idx.get(p_e2, -1)
            ))

        # 分组结构：{rel_type: [(gold_e1_idx, gold_e2_idx), ...]}
        gold_groups = defaultdict(list)
        for g_e1, g_e2, g_type in gold_rels:
            gold_groups[g_type].append((
                event_to_idx.get(g_e1, -1),
                event_to_idx.get(g_e2, -1)
            ))

        # --------------------------
        # 4. 矩阵化匹配计算
        # --------------------------
        tp = 0
        matched_gold = set()  # 存储已匹配的(g_type, g_idx)

        # 遍历每个关系类型
        for rel_type in set(pred_groups) | set(gold_groups):
            pred_pairs = pred_groups.get(rel_type, [])
            gold_pairs = gold_groups.get(rel_type, [])
            if not pred_pairs or not gold_pairs:
                continue

            # 构造相似度矩阵 (M_pred x M_gold)
            m = len(pred_pairs)
            n_gold = len(gold_pairs)
            match_scores = np.zeros((m, n_gold))

            # 批量计算匹配分数
            for i, (p_e1, p_e2) in enumerate(pred_pairs):
                if p_e1 == -1 or p_e2 == -1:
                    continue
                for j, (g_e1, g_e2) in enumerate(gold_pairs):
                    if g_e1 == -1 or g_e2 == -1:
                        continue
                    # 事件1和事件2的相似度需同时达标
                    e1_sim = sim_matrix[p_e1, g_e1]
                    e2_sim = sim_matrix[p_e2, g_e2]
                    match_scores[i,j] = min(e1_sim, e2_sim)

            # 找到最佳匹配（每个预测关系匹配最相似的未匹配标注）
            for i in range(m):
                best_j = -1
                best_score = 0
                for j in range(n_gold):
                    if (rel_type, j) in matched_gold:
                        continue
                    if match_scores[i,j] >= 0.8 and match_scores[i,j] > best_score:
                        best_score = match_scores[i,j]
                        best_j = j
                if best_j != -1:
                    tp += 1
                    matched_gold.add((rel_type, best_j))

        # --------------------------
        # 5. 计算最终指标
        # --------------------------
        fp = len(pred_rels) - tp
        fn = len(gold_rels) - len(matched_gold)

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall + 1e-9)
        # 数量差异惩罚（防负分）
        count_diff = abs(len(pred_rels) - len(gold_rels)) / max(len(gold_rels), 1)
        f1 *= (1 - 0.2 * min(count_diff, 1.0))

        rewards.append(max(f1, 0))

    return rewards


def normalize_event_str(event_str: str) -> str:
    """标准化事件五元组格式"""
    # 1. 去除首尾括号和空白
    cleaned = event_str.strip("()").strip()
    # 2. 统一分号格式
    cleaned = re.sub(r"\s*；\s*", "；", cleaned)
    # 3. 排序字段内容（可选，根据数据特征决定）
    parts = cleaned.split("；")
    if len(parts) == 5:
        # 对主语、宾语等字段排序（如"小狗，小男孩" -> "小男孩，小狗"）
        for i in [1, 2]:  # 主语(1)和宾语(2)字段
            if "，" in parts[i]:
                parts[i] = "，".join(sorted(parts[i].split("，")))
    return "；".join(parts)

<a name="Train"></a>
### Train the model

Now set up GRPO Trainer and all configurations!

In [ ]:
from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    use_vllm = True, # use vLLM for fast inference!
    learning_rate = 1e-5,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    logging_steps = 10,
    bf16 = is_bfloat16_supported(),
    fp16 = not is_bfloat16_supported(),
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 2, # Increase to 4 for smoother training
    num_generations = 4, # Decrease if out of memory
    max_prompt_length = 360,
    max_completion_length = 4000,
    num_train_epochs = 1, # Set to 1 for a full training run
    max_steps = 250,
    save_steps = 250,
    max_grad_norm = 0.1,
    report_to = "none", # Can use Weights & Biases
    output_dir = "outputs",
)

Unsloth: We now expect `per_device_train_batch_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 4


And let's run the trainer! If you scroll up, you'll see a table of rewards. The goal is to see the `reward` column increase!

You might have to wait 150 to 200 steps for any action. You'll probably get 0 reward for the first 100 steps. Please be patient!

| Step | Training Loss | reward    | reward_std | completion_length | kl       |
|------|---------------|-----------|------------|-------------------|----------|
| 1    | 0.000000      | 0.125000  | 0.000000   | 200.000000        | 0.000000 |
| 2    | 0.000000      | 0.072375  | 0.248112   | 200.000000        | 0.000000 |
| 3    | 0.000000      | -0.079000 | 0.163776   | 182.500000        | 0.000005 |


In [ ]:
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        format_reward_func,
        field_accuracy_reward_func_optimized,
        relation_accuracy_reward_optimized
    ],
    args = training_args,
    train_dataset = dataset_mini,
)

In [ ]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100 | Num Epochs = 5 | Total steps = 250
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 5,046,272/7,000,000,000 (0.07% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completion_length,kl,rewards / format_reward_func,rewards / field_accuracy_reward_func_optimized,rewards / relation_accuracy_reward_optimized
10,0.000000,0.902786,0.317168,1938.750000,0.000539,0.757646,0.140234,0.004906
20,0.000000,1.028311,0.173301,1521.862500,0.000795,0.854244,0.162308,0.011759


KeyboardInterrupt: 

<a name="Inference"></a>
### Inference
Now let's try the model we just trained! First, let's first try the model without any GRPO trained:

In [ ]:
text = tokenizer.apply_chat_template([
    {"role" : "user", "content" : "第二天早上小狗和小男孩发现瓶子里的青蛙不见了."},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 128,
)
output = model.fast_generate(
    [text],
    sampling_params = sampling_params,
    lora_request = None,
)[0].outputs[0].text

output

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  1.36it/s, est. speed input: 55.96 toks/s, output: 103.72 toks/s]


'从你描述的情景来看，小狗和小男孩很可能是发生了某种不幸的事情。青蛙被偷走了。这可能是因为他们无法找回，可能是被其他小动物偷走，或者是被主人不小心遗失了。无论如何，青蛙的消失显然给小男孩和他的家人带来了一定的困扰。在这样的情况下，小男孩和他的家人可能会感到非常担心和困惑。'

And now with the LoRA we just trained with GRPO - we first save the LoRA first!

In [ ]:
model.save_lora("grpo_saved_lora")

Now we load the LoRA and test:

In [ ]:
text = tokenizer.apply_chat_template([
    {"role" : "system", "content" : SYSTEM_PROMPT},
    {"role" : "user", "content" : "那个蜜蜂在蜂窝里飞了出来."},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 128,
)
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_saved_lora"),
)[0].outputs[0].text

output

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  3.28it/s, est. speed input: 581.44 toks/s, output: 39.42 toks/s]


'(飞；蜜蜂；无；无；无)'

**保存推理结果**

In [ ]:
def run_inference(input_file, output_file, model, tokenizer):
    # 读取eval.jsonl文件
    with open(input_file, 'r', encoding='utf-8') as infile, open(output_file, 'w', encoding='utf-8') as outfile:
        for line_num, line in enumerate(infile, start=1):
            # 解析每一行的JSON数据
            data = json.loads(line)
            text = data.get('text', '')

            # 生成text
            text_input = tokenizer.apply_chat_template([
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": text},
            ], tokenize=False, add_generation_prompt=True)

            # 设置采样参数
            sampling_params = SamplingParams(
                temperature=0.8,
                top_p=0.95,
                max_tokens=128,
            )

            # 推理并获得输出
            output = model.fast_generate(
                text_input,
                sampling_params=sampling_params,
                lora_request=model.load_lora("grpo_saved_lora"),
            )[0].outputs[0].text

            # 创建新的输出数据
            result = {
                'id': line_num,
                'input_text': text,
                'output': output
            }

            # 将结果写入到新的jsonl文件中
            outfile.write(json.dumps(result, ensure_ascii=False) + '\n')


In [ ]:
run_inference(input_file="/content/output_eval_merge.jsonl", output_file="output.jsonl", model=model, tokenizer=tokenizer)

[large output cleared]


Our reasoning model is much better - it's not always correct, since we only trained it for an hour or so - it'll be better if we extend the sequence length and train for longer!

<a name="Save"></a>
### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False: model.save_pretrained_merged("model", tokenizer, save_method = "lora",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "lora", token = "")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("model", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "hf/model", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "",
    )

Now, use the `model-unsloth.gguf` file or `model-unsloth-Q4_K_M.gguf` file in llama.cpp or a UI based system like Jan or Open WebUI. You can install Jan [here](https://github.com/janhq/jan) and Open WebUI [here](https://github.com/open-webui/open-webui)

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>
